# Build RAG Index

This notebook starts the RAG pipeline for Mohamad's portfolio documents.

In [ ]:
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

PROJECT_ROOT = Path.cwd().parent
RAW_DOCUMENTS_DIR = PROJECT_ROOT / "data" / "raw_documents"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma" / "virtual_twin"
BASE_MODEL_DIR = PROJECT_ROOT / "models" / "base" / "mlx-community__Qwen3-4B-4bit"
ADAPTER_DIR = PROJECT_ROOT / "adapters" / "virtual_twin_style"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Raw documents:", RAW_DOCUMENTS_DIR)
print("Vector DB:", CHROMA_DIR)
print("Embedding model:", EMBEDDING_MODEL)
print("Base model:", BASE_MODEL_DIR)
print("LoRA adapter:", ADAPTER_DIR)

In [ ]:
pdf_paths = sorted(RAW_DOCUMENTS_DIR.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {RAW_DOCUMENTS_DIR}")

for path in pdf_paths:
    print(path.name)

## Load Fine-Tuned Model

Load the local MLX base model together with the saved LoRA adapter.

In [ ]:
if not BASE_MODEL_DIR.exists():
    raise FileNotFoundError(f"Base model not found: {BASE_MODEL_DIR}")

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"LoRA adapter not found: {ADAPTER_DIR}")

model, tokenizer = load(str(BASE_MODEL_DIR), adapter_path=str(ADAPTER_DIR))
sampler = make_sampler(temp=0.5)

print("Fine-tuned model loaded with LoRA adapter.")

## Load PDF Documents

This function loads every PDF in `data/raw_documents/` as LangChain documents and keeps source metadata attached.

In [ ]:
def load_pdf_documents(raw_documents_dir: Path):
    pdf_paths = sorted(raw_documents_dir.glob("*.pdf"))

    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in {raw_documents_dir}")

    documents = []

    for pdf_path in pdf_paths:
        loader = PyPDFLoader(str(pdf_path))
        loaded_pages = loader.load()

        for page in loaded_pages:
            page.metadata["source_file"] = pdf_path.name
            page.metadata["source_path"] = str(pdf_path)

        documents.extend(loaded_pages)
        print(f"Loaded {len(loaded_pages)} pages from {pdf_path.name}")

    return documents


documents = load_pdf_documents(RAW_DOCUMENTS_DIR)
print(f"Total loaded pages: {len(documents)}")

## Split Documents Into Chunks

Qwen3 supports a large context window, but RAG works better with focused chunks. These settings create chunks that are small enough for accurate retrieval while leaving room for several chunks, the system prompt, chat history, and the final answer.

In [ ]:
def split_documents_into_chunks(
    documents,
    chunk_size: int = 900,
    chunk_overlap: int = 150,
):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = text_splitter.split_documents(documents)

    for index, chunk in enumerate(chunks):
        chunk.metadata["chunk_index"] = index
        chunk.metadata["chunk_size"] = len(chunk.page_content)

    return chunks


chunks = split_documents_into_chunks(documents)

print(f"Source pages: {len(documents)}")
print(f"Created chunks: {len(chunks)}")
print("First chunk source:", chunks[0].metadata.get("source_file"))
print("First chunk chars:", len(chunks[0].page_content))
print(chunks[0].page_content[:500])

## Create Embeddings

This creates the local embedding model used to convert chunks into vectors.

In [ ]:
def create_embedding_model(model_name: str = EMBEDDING_MODEL):
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


embedding_model = create_embedding_model()
test_vector = embedding_model.embed_query("What projects has Mohamad worked on?")

print("Embedding model loaded:", EMBEDDING_MODEL)
print("Embedding dimensions:", len(test_vector))

## Store Chunks In Vector DB

This stores the chunk embeddings in a persistent local Chroma vector database.

In [ ]:
def store_chunks_in_vector_db(
    chunks,
    embeddings,
    persist_directory: Path = CHROMA_DIR,
    collection_name: str = "virtual_twin_documents",
):
    if not chunks:
        raise ValueError("No chunks provided for vector storage.")

    persist_directory.mkdir(parents=True, exist_ok=True)

    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=str(persist_directory),
    )

    return vector_db


vector_db = store_chunks_in_vector_db(chunks, embedding_model)

print("Stored chunks:", len(chunks))
print("Chroma path:", CHROMA_DIR)

## Create Retriever

This creates a retriever from the Chroma vector database and adds a helper function for testing retrieval quality with source metadata.

In [ ]:
def create_retriever(
    vector_store,
    search_type: str = "mmr",
    k: int = 4,
    fetch_k: int = 12,
):
    return vector_store.as_retriever(
        search_type=search_type,
        search_kwargs={
            "k": k,
            "fetch_k": fetch_k,
        },
    )


retriever = create_retriever(vector_db)
print("Retriever ready.")

In [ ]:
def retrieve_context(query: str, max_preview_chars: int = 500):
    retrieved_docs = retriever.invoke(query)

    print(f"Query: {query}")
    print(f"Retrieved chunks: {len(retrieved_docs)}")

    for index, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page", "unknown")
        chunk_index = doc.metadata.get("chunk_index", "unknown")

        print(f"\n--- Chunk {index} ---")
        print(f"Source: {source} | Page: {page} | Chunk: {chunk_index}")
        print(doc.page_content[:max_preview_chars])

    return retrieved_docs


retrieved_docs = retrieve_context("What are Mohamad's main skills and experience?")

## Build RAG Answer Chain

This retrieves relevant chunks, builds a grounded prompt, and uses the local fine-tuned MLX model to format the final answer.

In [ ]:
RAG_SYSTEM_MESSAGE = (
    "You are Mohamad's professional virtual assistant. "
    "Answer using only the provided context. "
    "If the context is not enough, say that clearly and ask one concise follow-up question. "
    "Do not invent details. Do not pretend to be Mohamad. "
    "Keep the answer concise, helpful, and professional."
)


def format_retrieved_context(docs):
    context_blocks = []

    for index, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_file", "unknown source")
        page = doc.metadata.get("page", "unknown page")
        text = doc.page_content.strip()

        context_blocks.append(
            f"[Source {index}: {source}, page {page}]\n{text}"
        )

    return "\n\n".join(context_blocks)


def build_rag_messages(query: str, docs):
    context = format_retrieved_context(docs)

    user_message = (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )

    return [
        {"role": "system", "content": RAG_SYSTEM_MESSAGE},
        {"role": "user", "content": user_message},
    ]

In [ ]:
def answer_with_rag(query: str, max_tokens: int = 256):
    retrieved_docs = retriever.invoke(query)

    if not retrieved_docs:
        return {
            "answer": "I do not have enough context to answer that. Can you tell me which part of Mohamad's background or work you mean?",
            "sources": [],
        }

    rag_messages = build_rag_messages(query, retrieved_docs)
    prompt = tokenizer.apply_chat_template(
        rag_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    answer = generate(
        model,
        tokenizer,
        prompt=prompt,
        sampler=sampler,
        max_tokens=max_tokens,
    )

    sources = [
        {
            "source_file": doc.metadata.get("source_file"),
            "page": doc.metadata.get("page"),
            "chunk_index": doc.metadata.get("chunk_index"),
        }
        for doc in retrieved_docs
    ]

    return {"answer": answer, "sources": sources, "documents": retrieved_docs}


rag_result = answer_with_rag("What are Mohamad's main skills and experience?")
print(rag_result["answer"])
print("\nSources:")
for source in rag_result["sources"]:
    print(source)

## Interactive RAG Chat Test

Type a question and the notebook will retrieve context, generate an answer, and print the sources. Type `exit` or `quit` to stop.

In [ ]:
while True:
    user_query = input("Ask Mohamad's assistant: ").strip()

    if user_query.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    if not user_query:
        continue

    result = answer_with_rag(user_query)

    print("\nAssistant:")
    print(result["answer"])

    print("\nSources:")
    for source in result["sources"]:
        print(
            f"- {source.get('source_file')} "
            f"page={source.get('page')} "
            f"chunk={source.get('chunk_index')}"
        )

    print("-" * 80)